# 01 — Generate AUDIO-grounded Tool-Calling Data

**Run on a GPU Modal notebook** (Mimi encoding needs GPU; TTS is CPU/network).

The key fix over text-only training: the user's question is placed in the
**audio** rows of the code tensor, so Moshi learns to emit `<|tool_call|>` when
it *hears* a request — which is what actually happens at inference.

Per example we build `codes[17, T]`:

| rows | stream | content |
|------|--------|---------|
| `0` | text monologue | PAD while listening, then `<|tool_call|>…` + spoken reply |
| `1:9` | Moshi audio | silence (Mimi-encoded) |
| `9:17` | **user audio** | **edge-tts speech of the question, Mimi-encoded** |

Pipeline: `gen_tool_data.examples()` → edge-tts (many voices) → Mimi encode →
`codes[17,T]` → cache as one `.pt` on HuggingFace. Notebook 02 loads it and trains.

In [ ]:
# Install into the EXACT python running this kernel.
# datasets<3.0 encodes audio via soundfile (no torchcodec/ffmpeg) -> playable Audio column.
import sys
!{sys.executable} -m pip install edge-tts sphn nest_asyncio sentencepiece huggingface_hub einops numpy tqdm "datasets<3.0" soundfile -q
import importlib
for m in ["edge_tts", "sphn", "nest_asyncio", "sentencepiece", "einops",
          "huggingface_hub", "tqdm", "datasets", "soundfile"]:
    importlib.import_module(m)
import datasets
print(f"datasets {datasets.__version__}  ✓ (must be <3.0)")
print("If datasets was already imported as 3.x in this kernel, RESTART the kernel now.")

In [ ]:
import sys, gc, os, shutil, subprocess, asyncio, tempfile
from pathlib import Path
import numpy as np
import torch

# ── Clone repo ────────────────────────────────────────────────────────────────
REPO = Path("/repo")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone",
    "https://github.com/syedfahimabrar/moshi-D-gu.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "moshi"))
sys.path.insert(0, str(REPO / "notebooks"))

import sentencepiece as spm
from moshi.models import loaders
import gen_tool_data as G

# ── Config ────────────────────────────────────────────────────────────────────
HF_PATCHED_REPO = "abrarfahim/moshi-tool-patched"   # tokenizer source
HF_DATA_REPO    = "abrarfahim/moshi-tool-audio"     # where we cache the dataset
HF_TOKEN        = os.environ.get("HF_TOKEN", None)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SR     = 24000          # Mimi sample rate
FRAME  = 1920          # samples per 12.5 Hz frame (24000 / 12.5)

# edge-tts voices — variety of accents/genders so triggering generalises
VOICES = [
    "en-US-AriaNeural", "en-US-GuyNeural", "en-US-JennyNeural",
    "en-US-MichelleNeural", "en-US-ChristopherNeural", "en-US-EricNeural",
    "en-GB-SoniaNeural", "en-GB-RyanNeural", "en-GB-LibbyNeural",
    "en-AU-NatashaNeural", "en-AU-WilliamNeural",
    "en-IN-NeerjaNeural", "en-IN-PrabhatNeural",
    "en-CA-ClaraNeural", "en-CA-LiamNeural",
]
print(f"Device: {DEVICE} | voices: {len(VOICES)}")

In [ ]:
from huggingface_hub import hf_hub_download

# Tokenizer (patched, 32004 tokens)
tok_path = hf_hub_download(HF_PATCHED_REPO, "tokenizer_spm_32k_3.model", token=HF_TOKEN)
tok = spm.SentencePieceProcessor(); tok.Load(tok_path)
assert tok.get_piece_size() == 32004, tok.get_piece_size()
assert tok.id_to_piece(G.TOOL_CALL_ID) == "<|tool_call|>"
print(f"Tokenizer ✓ ({tok.get_piece_size()} tokens)")

# Mimi audio codec (from PersonaPlex), 8 codebooks
mimi_path = hf_hub_download(loaders.DEFAULT_REPO, loaders.MIMI_NAME, token=HF_TOKEN)
mimi = loaders.get_mimi(mimi_path, device=DEVICE)
mimi.eval()

# Verify N*FRAME samples -> N frames, and 8 codebooks
with torch.no_grad():
    probe = mimi.encode(torch.zeros(1, 1, FRAME * 10, device=DEVICE))
print(f"Mimi ✓  encode(10 frames) -> {tuple(probe.shape)}  (expect [1, 8, 10])")
assert probe.shape[1] == 8

In [ ]:
import edge_tts, sphn, nest_asyncio
nest_asyncio.apply()   # allow asyncio.run inside the Jupyter event loop

def _resample(wav, src, dst):
    if src == dst:
        return wav.astype("float32")
    n = int(round(len(wav) * dst / src))
    x  = np.linspace(0, 1, len(wav), endpoint=False)
    xn = np.linspace(0, 1, n, endpoint=False)
    return np.interp(xn, x, wav).astype("float32")

_tts_cache = {}   # (text, voice) -> waveform

async def _fetch_one(text, voice, sem, retries=3):
    async with sem:
        for attempt in range(retries):
            try:
                with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as fh:
                    path = fh.name
                await edge_tts.Communicate(text, voice).save(path)
                wav, sr = sphn.read(path); os.unlink(path)
                if wav.ndim > 1:
                    wav = wav.mean(axis=0)
                _tts_cache[(text, voice)] = _resample(wav, sr, SR)
                return
            except Exception:
                if attempt == retries - 1:
                    raise
                await asyncio.sleep(1.0)

async def _prefetch(pairs, concurrency=16):
    sem = asyncio.Semaphore(concurrency)
    todo = [(t, v) for (t, v) in pairs if (t, v) not in _tts_cache]
    for i in range(0, len(todo), 200):                 # chunk to show progress
        chunk = todo[i:i + 200]
        await asyncio.gather(*[_fetch_one(t, v, sem) for (t, v) in chunk])
        print(f"  TTS {min(i+200, len(todo))}/{len(todo)}")

def prefetch_tts(pairs, concurrency=16):
    """Fetch many (text, voice) clips CONCURRENTLY into the cache (the big speedup)."""
    asyncio.run(_prefetch(list(pairs), concurrency))

def tts_wav(text, voice):
    key = (text, voice)
    if key not in _tts_cache:                          # fallback for stragglers
        asyncio.run(_fetch_one(text, voice, asyncio.Semaphore(1)))
    return _tts_cache[key]

# smoke test
prefetch_tts([("what time is it", VOICES[0])])
print(f"TTS ✓  cached={len(_tts_cache)}  ({len(tts_wav('what time is it', VOICES[0]))} samples)")

In [ ]:
def _pad_to_frames(wav):
    """Pad a waveform up to a whole number of frames; return (wav, n_frames)."""
    n = len(wav)
    fr = int(np.ceil(n / FRAME)) if n > 0 else 0
    out = np.zeros(fr * FRAME, dtype="float32")
    out[:n] = wav
    return out, fr

def _silence(frames):
    return np.zeros(frames * FRAME, dtype="float32")

@torch.no_grad()
def _encode(wav):
    t = torch.from_numpy(wav).to(DEVICE)[None, None]   # [1,1,N]
    return mimi.encode(t)[0].cpu()                      # [8, T]

# Encode a long silence ONCE and slice it — avoids re-encoding silence per example.
_SIL = {"codes": None}
def _silence_codes(T):
    c = _SIL["codes"]
    if c is None or c.shape[1] < T:
        _SIL["codes"] = _encode(_silence(max(T + 8, 1100)))
    return _SIL["codes"][:, :T].clone()

def _fit(codes, T):
    """Trim or pad (repeat last/silence frame) audio codes to exactly T frames."""
    Ta = codes.shape[1]
    if Ta < T:
        codes = torch.cat([codes, codes[:, -1:].repeat(1, T - Ta)], dim=1)
    elif Ta > T:
        codes = codes[:, :T]
    return codes

def render(ex, voice):
    """Abstract Example -> dataset record (codes + mask + inspectable fields)."""
    user_parts, text, mask = [], [], []
    queries, replies = [], []
    for turn in ex.turns:
        if turn.query is None:                         # pure-silence turn
            N = int(np.random.randint(20, 50))
            k = int(np.random.randint(N // 3, N // 2))
            text += [G.PAD_ID] * N
            mask += [0] * k + [1] * (N - k)            # train tail to stay PAD
            user_parts.append(_silence(N))
            continue
        qwav, fq = _pad_to_frames(tts_wav(turn.query, voice))
        gap = int(np.random.randint(2, 6))
        emit_t, emit_m = G.render_emit(turn, tok)
        E = len(emit_t)
        text += [G.PAD_ID] * (fq + gap) + emit_t       # listen, pause, then emit
        mask += [0] * (fq + gap) + emit_m
        user_parts.append(np.concatenate([qwav, _silence(gap + E)]))
        queries.append(turn.query)
        replies.append(turn.reply)

    user_wav = np.concatenate(user_parts) if user_parts else _silence(1)
    T = len(text)
    user_codes  = _fit(_encode(user_wav), T)
    moshi_codes = _silence_codes(T)                    # Moshi audio = cached silence

    codes = torch.zeros(17, T, dtype=torch.long)
    codes[0]    = torch.tensor(text, dtype=torch.long)
    codes[1:9]  = moshi_codes
    codes[9:17] = user_codes
    return {
        "type":  ex.type,
        "query": " | ".join(queries),
        "reply": " | ".join(replies),
        "audio_array": user_wav.astype("float32"),     # written to a WAV file in cell-6
        "codes": codes.to(torch.int32).tolist(),       # 17 x T (Arrow-friendly)
        "mask":  [int(m) for m in mask],
        "voice": voice,
    }

print("render helpers ready ✓ (silence codes cached)")

In [ ]:
from tqdm.auto import tqdm
import soundfile as sf, pickle

CACHE   = Path("/tmp/tool_records.pkl")   # survives kernel restarts (same container)
WAV_DIR = Path("/tmp/tool_wavs")

if CACHE.exists():
    records = pickle.load(open(CACHE, "rb"))
    print(f"Loaded {len(records)} cached records (delete {CACHE} to regenerate)")
else:
    exs = G.examples(seed=42)

    # 1) Prefetch ALL unique (query, voice) clips concurrently — the big speedup.
    pairs = set()
    for i, ex in enumerate(exs):
        v = VOICES[i % len(VOICES)]
        for turn in ex.turns:
            if turn.query:
                pairs.add((turn.query, v))
    print(f"Prefetching {len(pairs)} unique TTS clips concurrently ...")
    prefetch_tts(pairs, concurrency=16)

    # 2) Render (Mimi encode on GPU; TTS now served from cache)
    np.random.seed(42)
    records = []
    for i, ex in enumerate(tqdm(exs, desc="Mimi encode")):
        records.append(render(ex, VOICES[i % len(VOICES)]))

    # 3) Write WAVs (HF Audio feature reads files via soundfile)
    if WAV_DIR.exists():
        shutil.rmtree(WAV_DIR)
    WAV_DIR.mkdir(parents=True)
    for i, r in enumerate(records):
        p = str(WAV_DIR / f"{i:05d}.wav")
        sf.write(p, r.pop("audio_array"), SR)
        r["audio"] = p
    pickle.dump(records, open(CACHE, "wb"))
    print(f"Built + cached {len(records)} records")

from collections import Counter
print("Breakdown:", dict(Counter(r["type"] for r in records)))
print(f"Avg frames: {sum(len(r['mask']) for r in records)/len(records):.0f}  "
      f"Max frames: {max(len(r['mask']) for r in records)}")

In [ ]:
# Build a proper HF dataset with a PLAYABLE audio column, push, and add a card.
# Requires datasets<3.0 (encodes audio via soundfile, no torchcodec).
from datasets import Dataset, Features, Value, Sequence, Audio
from huggingface_hub import HfApi

features = Features({
    "type":  Value("string"),
    "query": Value("string"),
    "reply": Value("string"),
    "voice": Value("string"),
    "audio": Audio(sampling_rate=SR),                   # playable in the HF viewer
    "codes": Sequence(Sequence(Value("int32"))),       # 17 x T training cache
    "mask":  Sequence(Value("int8")),
})

ds = Dataset.from_list(records, features=features)
print(ds)
ds.push_to_hub(HF_DATA_REPO, private=True, token=HF_TOKEN)

# ── Dataset card (README.md) ──────────────────────────────────────────────────
from collections import Counter
counts = Counter(r["type"] for r in records)
rows = "\n".join(f"| `{k}` | {counts[k]} |" for k in sorted(counts))
card = f"""---
license: mit
task_categories:
- audio-classification
- automatic-speech-recognition
language:
- en
tags:
- moshi
- personaplex
- tool-calling
- function-calling
- speech
size_categories:
- 1K<n<10K
---

# Moshi Tool-Calling — Audio-Grounded Dataset

Audio-grounded training data that teaches **Moshi / PersonaPlex** to emit
tool-call special tokens in its inner-monologue **when it hears** a request.

Each row is a pre-built code tensor `codes[17, T]` aligned to 12.5 Hz frames:

| rows | stream | content |
|------|--------|---------|
| `0` | text monologue | `PAD` while listening, then `<|tool_call|>…<|tool_end|>` + spoken reply |
| `1:9` | Moshi audio | silence |
| `9:17` | **user audio** | the spoken question (edge-tts), Mimi-encoded |

`mask` marks the text tokens the model must **emit** (tool call + reply); the
injected `<|tool_result|>` block is context (mask 0).

## Columns
- `type` — example category (see table below)
- `query` — the spoken user request (text)
- `reply` — Moshi's spoken response (text)
- `voice` — edge-tts voice used
- `audio` — the user waveform, 24 kHz mono (playable)
- `codes` — `17 × T` int tensor (1 text + 8 Moshi-audio + 8 user-audio rows)
- `mask` — `T` training mask over the text stream

## Special tokens
`<|tool_call|>`=32000, `<|tool_end|>`=32001, `<|tool_result|>`=32002, `<|tool_result_end|>`=32003

## Tools
`get_time` and `get_weather <city>`.

## Composition ({len(records)} examples)

| type | count |
|------|------|
{rows}

## Generation
edge-tts ({len(VOICES)} voices) → Mimi encode → `codes[17,T]`.
Built by `notebooks/01_generate_data.ipynb` in the
[moshi-D-gu](https://github.com/syedfahimabrar/moshi-D-gu) repo.
"""

HfApi(token=HF_TOKEN).upload_file(
    path_or_fileobj=card.encode(), path_in_repo="README.md",
    repo_id=HF_DATA_REPO, repo_type="dataset", token=HF_TOKEN)

print(f"\nPushed + card → https://huggingface.co/datasets/{HF_DATA_REPO}")
print("Next: run notebook 02 (it loads this dataset and trains).")

In [ ]:
# Build a proper HF dataset with a PLAYABLE audio column and push to the Hub.
# Requires datasets<3.0 (encodes audio via soundfile, no torchcodec).
from datasets import Dataset, Features, Value, Sequence, Audio

features = Features({
    "type":  Value("string"),
    "query": Value("string"),
    "reply": Value("string"),
    "voice": Value("string"),
    "audio": Audio(sampling_rate=SR),                   # playable in the HF viewer
    "codes": Sequence(Sequence(Value("int32"))),       # 17 x T training cache
    "mask":  Sequence(Value("int8")),
})

ds = Dataset.from_list(records, features=features)
print(ds)

ds.push_to_hub(HF_DATA_REPO, private=True, token=HF_TOKEN)
print(f"\nPushed → https://huggingface.co/datasets/{HF_DATA_REPO}")
print("Next: run notebook 02 (it loads this dataset and trains).")